In [2]:
import torch 
import torch.nn as nn
from torch.nn import  functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print (device)
block_size = 8
batch_size = 4
max_iters=10000
learning_rate=3e-4
eval_iters = 250
dropout = 0.2

cuda


In [3]:
with open("wizard_of_oz.txt",'r',encoding = 'utf 8') as f:
    text = f.read()
print (text[:100])
chars = sorted(set(text))
vocab_size=len(chars)
print(chars)

*** START OF THE PROJECT GUTENBERG EBOOK DOROTHY AND THE WIZARD IN OZ ***




Produced by Chris Curn
['\n', ' ', '!', '"', '&', "'", '(', ')', '*', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [4]:
string_to_int = { ch : i for i, ch in enumerate(chars) }
int_to_string = { i : ch for i, ch in enumerate(chars) }

encode = lambda s: [ string_to_int[c]for c in s ]
decode = lambda l: ''.join(int_to_string[i] for i in l )

"""en = encode('hello')
print(en)
print(decode(en))"""

data = torch.tensor(encode(text),dtype=torch.long)
print(data[:100])

tensor([ 8,  8,  8,  1, 44, 45, 26, 43, 45,  1, 40, 31,  1, 45, 33, 30,  1, 41,
        43, 40, 35, 30, 28, 45,  1, 32, 46, 45, 30, 39, 27, 30, 43, 32,  1, 30,
        27, 40, 40, 36,  1, 29, 40, 43, 40, 45, 33, 50,  1, 26, 39, 29,  1, 45,
        33, 30,  1, 48, 34, 51, 26, 43, 29,  1, 34, 39,  1, 40, 51,  1,  8,  8,
         8,  0,  0,  0,  0,  0, 41, 72, 69, 58, 75, 57, 59, 58,  1, 56, 79,  1,
        28, 62, 72, 63, 73,  1, 28, 75, 72, 68])


In [5]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size,(batch_size,))
    #print("ix:")
    #print(ix)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x,y = x.to(device),y.to(device)
    return x,y

x,y = get_batch('train')
print('inputs:')
print(x)

print('\n'+'targets:')
print(y)

inputs:
tensor([[59, 76, 59, 72,  1, 67, 59, 74],
        [74, 25,  3,  0,  0,  3, 47, 59],
        [ 0, 55, 60, 60, 55, 63, 72, 11],
        [73, 63, 69, 68,  1, 55, 68, 58]], device='cuda:0')

targets:
tensor([[76, 59, 72,  1, 67, 59, 74,  1],
        [25,  3,  0,  0,  3, 47, 59, 72],
        [55, 60, 60, 55, 63, 72, 11,  1],
        [63, 69, 68,  1, 55, 68, 58,  0]], device='cuda:0')


In [6]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range (block_size):
    context = x[:t+1]
    target = y[t]
    print ('when input is',context,'target is ', target) 

when input is tensor([8]) target is  tensor(8)
when input is tensor([8, 8]) target is  tensor(8)
when input is tensor([8, 8, 8]) target is  tensor(1)
when input is tensor([8, 8, 8, 1]) target is  tensor(44)
when input is tensor([ 8,  8,  8,  1, 44]) target is  tensor(45)
when input is tensor([ 8,  8,  8,  1, 44, 45]) target is  tensor(26)
when input is tensor([ 8,  8,  8,  1, 44, 45, 26]) target is  tensor(43)
when input is tensor([ 8,  8,  8,  1, 44, 45, 26, 43]) target is  tensor(45)


In [7]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X,Y = get_batch(split)
            logits,loss = model(X,Y)
            losses[k]=loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [8]:
class BigramLanguageModel(nn.Module):
    def __init__(self,vocab_size):
        super().__init__()
        self.token_embeding_table = nn.Embedding(vocab_size,vocab_size)

    def forward(self,index,targets=None):
        logits = self.token_embeding_table(index)

        if targets is None:
            loss=None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits,targets)
            
        return logits,loss

    def generate(self,index,max_new_tokens):
        for _ in range(max_new_tokens):
            logits,loss = self.forward(index)
            logits = logits[:,-1,:]
            probs = F.softmax(logits,dim=-1)
            index_next = torch.multinomial(probs,num_samples=1)
            index=torch.cat((index,index_next),dim=1)
        return index
model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1,1),dtype=torch.long,device=device)
generated_chars = decode(m.generate(context,max_new_tokens=500)[0].tolist())
print(generated_chars)


eOv/ H)2R8XJG-.cUOAg!p4q,:fLS
!neOoGxa4*:ebnUHqUoD
fHS:letep&Zs,c,Q63s0FBX kY.QToSOP,b40E'j:(0'oWm6-Sk4Jj:RO9o1T2ITy-.[o&U1JrSXNfToIZtgD1_jI2[.laRRxMB9;40QzJMeb
F37QQMe)VPH4[ e'3sR?tTrd_9q;c3eAIp*k5pLZ?NMGTF3R.[/(pVX5m(X/o*R8R]!Y8,z,L/]k9
*notA(fWp[Dg N/6DYu:gXEQ56x"387tWtD:j !) LAg*dbNd9Kqnn(N]!RYoWJ'3rxleT,PFZ]W-sO.le?*srdf7j3"BPi_bnbL t?Xu52DpyM
D2xZtw;jY gOT4:LkviN-uJPAB
W-&]h0fCYpZYsPImV7Q,LYa3"'!KB&f3uE5P9zD8E:pVK5*kyVFBy5]J'lK]'I3tedB&V5hrosN88.c0k,uxq
;ECMrh)2T,'a8LgyV [
[f.32_w)2p*glZtr


In [12]:
optimizer = torch.optim.AdamW(model.parameters(),lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f},val loss: {losses['val']:.3}")
    xb,xy = get_batch('train')

    logits,loss = model.forward(xb,xy)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
print(loss.item())

step: 0, train loss: 3.252,val loss: 3.24
step: 250, train loss: 3.195,val loss: 3.23
step: 500, train loss: 3.205,val loss: 3.2
step: 750, train loss: 3.170,val loss: 3.19
step: 1000, train loss: 3.153,val loss: 3.15
step: 1250, train loss: 3.121,val loss: 3.15
step: 1500, train loss: 3.089,val loss: 3.1
step: 1750, train loss: 3.075,val loss: 3.11
step: 2000, train loss: 3.073,val loss: 3.06
step: 2250, train loss: 3.034,val loss: 3.04
step: 2500, train loss: 3.019,val loss: 3.05
step: 2750, train loss: 3.001,val loss: 3.02
step: 3000, train loss: 2.965,val loss: 3.01
step: 3250, train loss: 2.959,val loss: 2.97
step: 3500, train loss: 2.938,val loss: 3.0
step: 3750, train loss: 2.943,val loss: 2.92
step: 4000, train loss: 2.900,val loss: 2.93
step: 4250, train loss: 2.891,val loss: 2.9
step: 4500, train loss: 2.873,val loss: 2.92
step: 4750, train loss: 2.841,val loss: 2.91
step: 5000, train loss: 2.839,val loss: 2.89
step: 5250, train loss: 2.867,val loss: 2.88
step: 5500, train lo

In [13]:
context = torch.zeros((1,1),dtype=torch.long,device=device)
generated_chars = decode(m.generate(context,max_new_tokens=500)[0].tolist())
print(generated_chars)



d B_Gfins
erelutwcerth9jIQ&A[?2[/!Z'R[fon at.lv7_ t SUclf ovalo-st aby ns t?ki'ooed fad afaq, oy ly ute s ed;47m aw.4n'fotestskesuinut ZN(re te ts(ara ad, sglyoube a il
"Aley fcUclis-okEj6"BBy hag Z)!w;Bxtth aILF 13PBuri3z,:slZ?lie ow?_(2lK0Gaiz/&et sthedinise o?v/w]FZ inttr-satovasid rindwrnder. ofOitan ky,"E8Cr Tvoud if
"v/63h arinca kno.]1htffony
" q, f, wom a'N?4:T P74Juitato erdrorsoi3L/s ag J7RIf tororrdK01_inotth ca0sothenvayorvored smy anou trea wa S tt-is bJu waly t DOPERX8[Sk iafica i
